In [29]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf
import opendatasets as od 
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import pandas as pd
import os

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn import tree
import random
from PIL import Image
import cv2

from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout



In [2]:
# od.download(
#     "https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k")

data = pd.read_excel("Processed_data.xlsx")
path_to_images = "./ocular-disease-recognition-odir5k/"
training_path = "ODIR-5k/ODIR-5k/Training Images/"

In [3]:
# Creating the target Labels 
def createTarget(df):
    df['Labels'] = [[0,0,0,0,0,0,0] for i in range (0,len(df))] #create a column for labels with 8 0s
    target_columns = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
    label = [] #create empty label array
    for i in range(0, len(df)):
        for target in target_columns:
            label.append(df.loc[i, target]) #append the value for the column for each row to the label
        
        df.at[i,'Labels'] = np.array(label) #update the label column with the label array
        label = [] #reset label array
            

In [4]:
createTarget(data)

In [5]:
data = data.drop(['Patient ID', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O'], axis=1)
data

,Patient Age,Patient Sex,Filename,Diagnosis,Labels
0,69,Female,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,57,Male,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy","[0, 1, 0, 0, 0, 0, 0, 1]"
3,66,Male,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,53,Male,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]"
...,...,...,...,...,...
6995,63,Male,4686_right.jpg,proliferative diabetic retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6996,42,Male,4688_right.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6997,54,Male,4689_right.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
6998,57,Male,4690_right.jpg,mild nonproliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"


In [6]:
# Path to your image dataset
image_dataset_path = "./ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training images"
validation_path = "./ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Testing images"

In [7]:
data["filepath"] = data['Filename'].apply(lambda x: os.path.join(image_dataset_path, x))

In [14]:
from PIL import Image
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Function to preprocess images
def preprocess_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = image.resize((128, 128))  # Resize to the desired input size for your CNN
    return np.array(image,dtype=np.float32)

# Applying the function to each image path
data['image'] = data['filepath'].apply(preprocess_image)

In [15]:
# Normalize age and encode gender
age_scaler = MinMaxScaler()
ages_scaled =  age_scaler.fit_transform(data[['Patient Age']])
data['Patient Sex'] = data['Patient Sex'].map({'Male': 0, 'Female': 1})  # Binary encoding

In [ ]:
# from sklearn.preprocessing import OneHotEncoder
# import numpy as np


# # Encode gender
# encoder = OneHotEncoder()
# genders_encoded = encoder.fit_transform(data[['Patient Sex']]).toarray()

# # Combine age and gender
# age_gender_combined = np.hstack((ages_scaled, genders_encoded))

In [11]:
import tensorflow as tf
from tensorflow.keras import layers

class EyesCNN(tf.keras.Model):
    def __init__(self, dropout_rate=0.1):
        super(EyesCNN, self).__init__()

        self.conv1 = layers.Conv2D(32, (3, 3), strides=(1, 1), padding='same', activation='relu')
        self.conv2 = layers.Conv2D(64, (3, 3), strides=(1, 1), padding='same', activation='relu')
        self.conv3 = layers.Conv2D(128, (3, 3), strides=(1, 1), padding='same', activation='relu')
        self.conv4 = layers.Conv2D(128, (3, 3), strides=(1, 1), padding='same', activation='relu')
        self.conv5 = layers.Conv2D(256, (2, 2), strides=(2, 2), padding='same', activation='relu')
        self.conv6 = layers.Conv2D(128, (2, 2), strides=(2, 2), padding='same', activation='relu')

        # Define the max pooling layer
        self.pool = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid')

        # Define batch normalization layer
        self.bn = layers.BatchNormalization()

        # Define dropout layer
        self.dropout = layers.Dropout(rate=dropout_rate)

        # Define fully connected layers
        self.flatten = layers.Flatten()
        self.fc1 = layers.Dense(128, activation='relu')
        self.fc2 = layers.Dense(8, activation ='sigmoid')

    def call(self, inputs, age):
        # Apply depthwise separable convolution and pooling layers
        x = self.conv1(inputs)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.conv5(x)
        x = self.conv6(x)
        x = self.bn(x)
        x = self.pool(x)

        # Apply dropout
        x = self.dropout(x)

        # Reshape the tensor for fully connected layers
        x = self.flatten(x)

        # Concatenate age information to the flattened tensor
        age = tf.expand_dims(age, axis=0)
        age = tf.cast(age, dtype=x.dtype)
        x = tf.concat([x, age], axis=1)

        # Apply fully connected layers
        x = self.fc1(x)
        x = self.fc2(x)
        return x

# Instantiate the model
model = EyesCNN()


In [16]:
image_data = np.array(data["image"])

labels = np.array(data["Labels"].apply(lambda x: np.expand_dims(x, axis=0)))

In [17]:
len(labels)

7000

In [18]:
len(image_data)

7000

In [19]:
# train_images, val_images, train_age_gender, val_age_gender, train_labels, val_labels = train_test_split(
#     image_data, age_gender_combined, labels, test_size=0.2, random_state=42)

train_images, val_images, train_labels, val_labels = train_test_split(
    image_data, labels, test_size=0.2, random_state=42)

print(len(train_images))
print(len(val_images))
print(len(train_labels))
print(len(val_labels))

5600
1400
5600
1400


In [20]:
age_train, age_val = train_test_split(ages_scaled, random_state = 42, test_size = 0.2)

In [30]:
from tqdm import tqdm

In [23]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.003)
loss_fn = tf.keras.losses.BinaryCrossentropy()
metrics = ['accuracy']

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)

In [ ]:
# Train the model
num_epochs = 5
for epoch in range(num_epochs):
    tqdm_train_data = tqdm(zip(train_images, train_labels, age_train), total=len(train_images), desc=f'Epoch {epoch + 1}')
    for images, labels, age in tqdm_train_data:
        images = tf.expand_dims(images, axis=0)
        with tf.GradientTape() as tape:            
            predictions = model(images, age, training = True)
            loss = loss_fn(labels, predictions)

        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        
    tqdm_train_data.close()
    # Evaluate the model on validation data
    tqdm_val_data = tqdm(zip(val_images, val_labels, age_val), total=len(val_images), desc='Validation')

    val_loss_sum = 0.0
    correct_multi_label_predictions = 0
    total_samples = 0

    for images, labels, age in tqdm_val_data:
        images = tf.expand_dims(images, axis=0)
        predictions = model(images, age, training=False)
        loss = loss_fn(labels, predictions)
        val_loss_sum += loss.numpy()

        # Convert logits to predicted labels
        predicted_labels = tf.cast(tf.greater(predictions, 0.5), tf.int32)
        true_labels = labels

        # Update correct multi-label predictions count
        correct_multi_label_predictions += tf.reduce_sum(tf.cast(tf.equal(predicted_labels, true_labels), tf.int32)).numpy()

        total_samples += 1

        # Update the tqdm progress bar for validation
        tqdm_val_data.set_postfix({'loss': val_loss_sum / total_samples}, refresh=True)
        
    # Compute the average validation loss
    average_val_loss = val_loss_sum / total_samples

    # Compute multi-label accuracy
    accuracy = correct_multi_label_predictions / total_samples

    # Close the tqdm progress bar for validation
    tqdm_val_data.close()

    print(f'Epoch: {epoch}, Validation Loss: {average_val_loss}, Multi-Label Accuracy: {accuracy}')
    

Epoch 1: 100%|█████████████████████████████████████████████████████████████████████| 5600/5600 [26:31<00:00,  3.52it/s]
Validation: 100%|███████████████████████████████████████████████████████| 1400/1400 [01:17<00:00, 18.13it/s, loss=4.41]
Epoch: 0, Validation Loss: 4.408904375497784, Multi-Label Accuracy: 6.89
Epoch 2: 100%|█████████████████████████████████████████████████████████████████████| 5600/5600 [25:08<00:00,  3.71it/s]
Validation: 100%|██████████████████████████████████████████████████████| 1400/1400 [01:13<00:00, 19.02it/s, loss=0.328]
Epoch: 1, Validation Loss: 0.32817935935088566, Multi-Label Accuracy: 6.8914285714285715
Epoch 3: 100%|█████████████████████████████████████████████████████████████████████| 5600/5600 [26:01<00:00,  3.59it/s]
Validation: 100%|██████████████████████████████████████████████████████| 1400/1400 [01:14<00:00, 18.75it/s, loss=0.328]
Epoch: 2, Validation Loss: 0.32824578006352695, Multi-Label Accuracy: 6.8914285714285715
Epoch 4:  12%|████████▏                                                             | 650/5600 [02:49<21:32,  3.83it/s]